# DAY2 실습 06 : 의류 공장 생산성 데이터

부서·팀·구간으로 집단을 나누어 생산성과 목표 달성률을 비교하고, 두 수치의 관계를 부서별로 확인한다.
문제 아래 코드 셀에 직접 코드를 작성하고 실행해 본다.

데이터 파일 : `garments_worker_productivity.csv`
출처 : UCI Machine Learning Repository - [Productivity Prediction of Garment Employees](https://archive.ics.uci.edu/dataset/597/productivity+prediction+of+garment+employees)

의류 공장 생산팀의 2015년 1월 1일 ~ 3월 11일 기록이다. 한 줄이 날짜 하루 · 팀 하나 · 부서 하나의 기록이고 총 1,197행이다. 금요일 기록은 없다.
파일을 `HD-AI/data` 폴더에 넣은 뒤 실행한다.

| 컬럼 | 설명 |
|---|---|
| date | 날짜 |
| quarter | 한 달을 7일 단위로 나눈 구간 (Quarter1 : 1~7일 … Quarter5 : 29~31일). 분기가 아니다 |
| department | 부서 (sewing : 봉제 / finishing : 마감) |
| day | 요일 |
| team | 팀 번호 (1~12) |
| targeted_productivity | 팀·날짜별로 정한 목표 생산성 |
| smv | 표준작업시간 (Standard Minute Value). 작업 하나에 배정된 시간 (분) |
| wip | 재공품 수. 아직 완성되지 않은 제품 수 |
| over_time | 팀 초과근무 시간 (분) |
| incentive | 팀 인센티브 금액 (BDT, 방글라데시 타카) |
| idle_time | 생산이 중단된 시간 |
| idle_men | 생산 중단으로 작업하지 못한 인원 |
| no_of_style_change | 제품 스타일 변경 횟수 |
| no_of_workers | 팀 인원 |
| actual_productivity | 실제 생산성. 원 설명은 0~1 범위이며, 1을 넘는 기록도 37건 있다 |

In [ ]:
%pip install -q koreanize-matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
data_folder = "/content/drive/MyDrive/HD-AI/data"
csv_path = os.path.join(data_folder, "garments_worker_productivity.csv")

df = pd.read_csv(csv_path)

data = df.copy()

### 1. `value_counts` : 부서 이름 확인

상황 : 부서별로 비교하기 전에 그룹 기준을 확인한다. 부서는 sewing(봉제)과 finishing(마감) 두 개다.

원본 `df`의 `department` 값별 기록 수를 구하시오. 몇 개의 값으로 나오는지 확인한다.

In [ ]:
# 힌트 : df["열"].value_counts()




In [ ]:
# 제공 코드 : 부서 이름을 정리한다 ("sweing" → "sewing", 끝에 공백이 붙은 "finishing " → "finishing")
data["department"] = data["department"].replace({"sweing": "sewing", "finishing ": "finishing"})
data["department"].value_counts()

### 2. 분석 단위와 `size` / `count`

상황 : 공장장이 "부서별 재공품(`wip`) 평균을 비교해 달라"고 요청했다. 요청대로 비교할 수 있는지 먼저 확인한다.

1) 날짜·팀·부서 조합별 행 수의 최댓값을 구하시오.  한 행이 "날짜 하루·팀 하나·부서 하나"의 기록이다.

2) `department`별 행 수(size)와 `wip`의 유효값 수(count)를 한 표로 구하시오.

In [ ]:
# 힌트 : groupby(["열1", "열2", "열3"]).size().max(), groupby("열").size(), groupby("열")["열"].count()




### 3. 0/1 평균 = 비율 : 부서별 목표 달성률

상황 : "어느 부서가 더 잘하고 있는가?"를 평균 생산성과 목표 달성률, 두 기준으로 확인하려 한다.

1) `actual_productivity`가 `targeted_productivity` 이상이면 1, 아니면 0인 `달성` 열을 만드시오.

2) 전체 달성률(%)을 구하시오.

3) `department`별 n, 달성 건수, 달성률(%), 평균 생산성, 생산성 중앙값을 한 표로 구하시오.

In [ ]:
# 힌트 : (조건).astype(int), groupby("열").agg(n=("열", "size"), 달성건수=("열", "sum"), ...)




### 4. 박스플롯 : 부서별 생산성 분포

상황 : 3번에서 두 기준의 결과가 같은지 분포로 확인한다.

`department`별 `actual_productivity` 박스플롯을 그리시오.

In [ ]:
# 힌트 : sns.boxplot(data=, x=, y=)




### 5. `pivot_table` : 부서 × 구간 평균과 관측 수

상황 : 부서와 구간(`quarter`) 조합 중 생산성이 가장 높은 칸을 우수 사례로 보고하려 한다.

`department`를 행, `quarter`를 열로 하여 `actual_productivity`의 평균 피벗표와 관측 수 피벗표를 각각 구하시오.

In [ ]:
# 힌트 : pivot_table(index=, columns=, values=, aggfunc="mean"), aggfunc="count"




### 6. `rank` : 팀별 달성률 순위

상황 : 목표 달성률이 낮은 하위 2개 팀을 지원 대상으로 정하려 한다.

`team`별 n과 달성률(%)을 구하고, 달성률 순위를 붙여 순위 순으로 정렬하시오. 높은 값이 1위이고, 동점은 `method="min"`으로 처리한다.

In [ ]:
# 힌트 : groupby("team").agg(...), rank(ascending=False, method="min"), sort_values("열")


